# 03_meta_logreg_all_baseline_bigsearch_optuna_final

LogReg meta-feature screening на полном baseline-пуле моделей из `ML_razdel_pointfix_final.ipynb`.

Протокол:
- тот же внешний `80/20` chronological holdout;
- выбор logreg-схем идёт только по CV на `train_filt = train_full[duration_hours < 960]`;
- регрессоры здесь остаются baseline, без tuning;
- включена печать каждого прогона и сохранение промежуточного прогресса.

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import json
import time
import sys
import subprocess
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

from sklearn.linear_model import LogisticRegression, Lasso, Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering, Birch, DBSCAN, OPTICS, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import pairwise_distances

from xgboost import XGBRegressor

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

SEED = 2
TARGET_COL = "duration_hours"
DURATION_CAP = 960
HOLDOUT_FRAC = 0.20
CV_SPLITS = 3
LOGREG_OOF_SPLITS = 5
USE_PCA = True
PCA_COMPONENTS = 30

DATA_CANDIDATES = [
    Path("./data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
    Path("./data_finall.csv"),
    Path("/mnt/data/data_finall.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Не найден data_finall_without_TTM.csv / data_finall.csv")

ARTIFACT_DIR = Path("./artifacts_meta_all_baseline_bigsearch_optuna")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def normalized_entropy(proba):
    proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.zeros(len(proba), dtype=float)
    ent = -(proba * np.log(proba)).sum(axis=1)
    return ent / np.log(proba.shape[1])

def top2_margin(proba):
    proba = np.asarray(proba, dtype=float)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.ones(len(proba), dtype=float)
    desc = np.sort(proba, axis=1)[:, ::-1]
    return desc[:, 0] - desc[:, 1]

def json_ready(obj):
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [json_ready(v) for v in obj.tolist()]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    return obj

def load_split_data():
    df = pd.read_csv(DATA_PATH)
    split = int(len(df) * (1 - HOLDOUT_FRAC))

    train_full = df.iloc[:split].copy().reset_index(drop=True)
    holdout_full = df.iloc[split:].copy().reset_index(drop=True)

    train_filt = train_full[train_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)
    holdout_typical = holdout_full[holdout_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)

    return df, train_full, train_filt, holdout_full, holdout_typical

BASELINE_PIPELINES = [
    ("Scaled_Lasso", Pipeline([
        ("Scaler", StandardScaler()),
        ("Lasso", Lasso(random_state=SEED))
    ])),
    ("Scaled_Elastic", Pipeline([
        ("Scaler", StandardScaler()),
        ("Elastic", ElasticNet(random_state=SEED))
    ])),
    ("Scaled_RF_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("RF", RandomForestRegressor(random_state=SEED))
    ])),
    ("Scaled_ET_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("ET", ExtraTreesRegressor(random_state=SEED))
    ])),
    ("Scaled_BR_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("BR", BaggingRegressor(random_state=SEED))
    ])),
    ("Scaled_DT_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("DT_reg", DecisionTreeRegressor(random_state=SEED))
    ])),
    ("Scaled_Ridge", Pipeline([
        ("Scaler", StandardScaler()),
        ("Ridge", Ridge(random_state=SEED))
    ])),
    ("Scaled_SVR", Pipeline([
        ("Scaler", StandardScaler()),
        ("SVR", SVR(kernel="linear", C=1e2))
    ])),
    ("Scaled_Hub-Reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("Hub-Reg", HuberRegressor())
    ])),
    ("Scaled_BayRidge", Pipeline([
        ("Scaler", StandardScaler()),
        ("BR", BayesianRidge())
    ])),
    ("Scaled_KNN_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("KNN_reg", KNeighborsRegressor())
    ])),
    ("Scaled_Gboost-Reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("GBoost-Reg", GradientBoostingRegressor(random_state=SEED))
    ])),
    ("Scaled_XGB_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("XGBR", XGBRegressor(random_state=SEED))
    ])),
    ("Scaled_RFR_PCA", Pipeline([
        ("Scaler", StandardScaler()),
        ("PCA", PCA(n_components=3)),
        ("RF", RandomForestRegressor(random_state=SEED))
    ])),
    ("Scaled_XGBR_PCA", Pipeline([
        ("Scaler", StandardScaler()),
        ("PCA", PCA(n_components=3)),
        ("XGB", XGBRegressor(random_state=SEED))
    ])),
]

MODEL_ORDER = [name for name, _ in BASELINE_PIPELINES]
BASELINE_PIPELINE_LOOKUP = {name: model for name, model in BASELINE_PIPELINES}

def build_regression_model(model_name, extra_params=None):
    if model_name not in BASELINE_PIPELINE_LOOKUP:
        raise ValueError(model_name)

    est = clone(BASELINE_PIPELINE_LOOKUP[model_name])

    if extra_params is not None:
        current = est.get_params()
        safe_updates = {k: v for k, v in extra_params.items() if k in current}
        if safe_updates:
            est.set_params(**safe_updates)

    return est

def compute_baseline_cv_and_holdout(model_names, train_filt, holdout_typical, holdout_full):
    rows = []
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

    Xtrain = train_filt.drop(columns=[TARGET_COL]).reset_index(drop=True)
    ytrain = train_filt[TARGET_COL].reset_index(drop=True)

    Xhold_typ = holdout_typical.drop(columns=[TARGET_COL]).reset_index(drop=True)
    yhold_typ = holdout_typical[TARGET_COL].reset_index(drop=True)

    Xhold_full = holdout_full.drop(columns=[TARGET_COL]).reset_index(drop=True)
    yhold_full = holdout_full[TARGET_COL].reset_index(drop=True)

    for model_name in model_names:
        fold_maes = []

        for tr_idx, va_idx in tscv.split(train_filt):
            fold_train = train_filt.iloc[tr_idx].copy().reset_index(drop=True)
            fold_valid = train_filt.iloc[va_idx].copy().reset_index(drop=True)

            Xtr = fold_train.drop(columns=[TARGET_COL]).reset_index(drop=True)
            ytr = fold_train[TARGET_COL].reset_index(drop=True)

            Xva = fold_valid.drop(columns=[TARGET_COL]).reset_index(drop=True)
            yva = fold_valid[TARGET_COL].reset_index(drop=True)

            est = build_regression_model(model_name)
            est.fit(Xtr, ytr)
            pred = est.predict(Xva)
            fold_maes.append(float(mean_absolute_error(yva, pred)))

        est_full = build_regression_model(model_name)
        est_full.fit(Xtrain, ytrain)

        pred_typ = est_full.predict(Xhold_typ)
        pred_full = est_full.predict(Xhold_full)

        m_typ = reg_metrics(yhold_typ, pred_typ)
        m_full = reg_metrics(yhold_full, pred_full)

        rows.append({
            "model": model_name,
            "baseline_cv_MAE": round(float(np.mean(fold_maes)), 4),
            "baseline_cv_std": round(float(np.std(fold_maes)), 4),
            "baseline_holdout_typical_MAE": round(m_typ["MAE"], 4),
            "baseline_holdout_typical_RMSE": round(m_typ["RMSE"], 4),
            "baseline_holdout_typical_MdAE": round(m_typ["MdAE"], 4),
            "baseline_holdout_full_MAE": round(m_full["MAE"], 4),
            "baseline_holdout_full_RMSE": round(m_full["RMSE"], 4),
            "baseline_holdout_full_MdAE": round(m_full["MdAE"], 4),
        })

    return pd.DataFrame(rows).sort_values(["baseline_cv_MAE", "baseline_cv_std"]).reset_index(drop=True)

all_df, train_full, train_filt, holdout_full, holdout_typical = load_split_data()

baseline_cache_path = ARTIFACT_DIR / "baseline_summary_all_baseline_bigsearch_optuna.csv"
if baseline_cache_path.exists():
    baseline_summary = pd.read_csv(baseline_cache_path)
    baseline_summary = baseline_summary.sort_values(["baseline_cv_MAE", "baseline_cv_std"]).reset_index(drop=True)
    print("Loaded baseline summary from cache:", baseline_cache_path)
else:
    baseline_summary = compute_baseline_cv_and_holdout(MODEL_ORDER, train_filt, holdout_typical, holdout_full)
    baseline_summary.to_csv(baseline_cache_path, index=False)
    print("Computed and cached baseline summary:", baseline_cache_path)

print("DATA_PATH =", DATA_PATH)
print("all rows =", len(all_df))
print("train_full =", len(train_full))
print("train_filt (<960) =", len(train_filt))
print("holdout_full =", len(holdout_full))
print("holdout_typical (<960) =", len(holdout_typical))
print("ARTIFACT_DIR =", ARTIFACT_DIR)
print("baseline models =", len(MODEL_ORDER))
display(baseline_summary)


In [ ]:

BINNING_SCHEMES = {
    "2cl: 0-200 / 200+": {"mode": "fixed", "edges": [0, 200, np.inf], "labels": ["0-200h", "200+h"]},
    "2cl: 0-500 / 500+": {"mode": "fixed", "edges": [0, 500, np.inf], "labels": ["0-500h", "500+h"]},
    "2cl: median(train)": {"mode": "median", "labels": None},
    "3cl: 0-150 / 151-700 / 700+": {"mode": "fixed", "edges": [0, 150, 700, np.inf], "labels": ["0-150h", "151-700h", "700+h"]},
    "3cl: 0-100 / 101-500 / 500+": {"mode": "fixed", "edges": [0, 100, 500, np.inf], "labels": ["0-100h", "101-500h", "500+h"]},
    "3cl: 0-200 / 201-1000 / 1000+": {"mode": "fixed", "edges": [0, 200, 1000, np.inf], "labels": ["0-200h", "201-1000h", "1000+h"]},
    "4cl: 0-100 / 101-500 / 501-1500 / 1500+": {"mode": "fixed", "edges": [0, 100, 500, 1500, np.inf], "labels": ["0-100h", "101-500h", "501-1500h", "1500+h"]},
    "4cl: 0-200 / 201-700 / 701-2000 / 2000+": {"mode": "fixed", "edges": [0, 200, 700, 2000, np.inf], "labels": ["0-200h", "201-700h", "701-2000h", "2000+h"]},
    "4cl: quantile_4(train)": {"mode": "quantile", "q": 4, "labels": None},
    "5cl: 0-200 / 201-1000 / 1001-2000 / 2001-3000 / 3000+": {"mode": "fixed", "edges": [0, 200, 1000, 2000, 3000, np.inf], "labels": ["0-200h", "201-1000h", "1001-2000h", "2001-3000h", "3000+h"]},
    "5cl: 0-50 / 51-200 / 201-500 / 501-1500 / 1500+": {"mode": "fixed", "edges": [0, 50, 200, 500, 1500, np.inf], "labels": ["0-50h", "51-200h", "201-500h", "501-1500h", "1500+h"]},
    "5cl: quantile_5(train)": {"mode": "quantile", "q": 5, "labels": None},
    "6cl: 0-24 / 25-100 / 101-300 / 301-700 / 701-2000 / 2000+": {"mode": "fixed", "edges": [0, 24, 100, 300, 700, 2000, np.inf], "labels": ["0-24h", "25-100h", "101-300h", "301-700h", "701-2000h", "2000+h"]},
}
WEIGHT_ALPHAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

VERBOSE_EACH_RUN = True
PRINT_FOLD_VALUES = True
SAVE_PROGRESS_EVERY = 50

def fit_binning_on_train(y_train, scheme_cfg):
    mode = scheme_cfg["mode"]

    if mode == "fixed":
        return {"edges": list(scheme_cfg["edges"]), "labels": list(scheme_cfg["labels"])}

    if mode == "median":
        med = float(y_train.median())
        return {
            "edges": [float(y_train.min()) - 1e-9, med, np.inf],
            "labels": [f"<= {med:.0f}", f"> {med:.0f}"],
        }

    if mode == "quantile":
        _, edges = pd.qcut(y_train, q=scheme_cfg["q"], retbins=True, duplicates="drop")
        edges = list(edges)
        edges[0] = float(min(edges[0], y_train.min() - 1e-9))
        edges[-1] = np.inf
        labels = [f"class_{i}" for i in range(len(edges) - 1)]
        return {"edges": edges, "labels": labels}

    raise ValueError(mode)

def apply_binning(y, fitted_binning):
    cls = pd.cut(
        y,
        bins=fitted_binning["edges"],
        labels=False,
        include_lowest=True,
        right=True,
    )
    if cls.isna().any():
        raise RuntimeError(f"NaN after binning: {int(cls.isna().sum())}")
    return cls.astype(int).values

def compute_soft_class_weight(y_cls, alpha):
    counts = pd.Series(y_cls).value_counts().sort_index()
    n = len(y_cls)
    k = len(counts)

    weights = {}
    for c, cnt in counts.items():
        balanced = n / (k * cnt)
        weights[int(c)] = float(1.0 + alpha * (balanced - 1.0))
    return weights

def build_logreg_classifier(alpha, y_cls):
    class_weight = compute_soft_class_weight(y_cls, alpha)
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", LogisticRegression(
            random_state=SEED,
            max_iter=4000,
            solver="lbfgs",
            class_weight=class_weight,
        ))
    ])

def aligned_proba(clf, X, n_classes):
    raw = clf.predict_proba(X)
    out = np.zeros((len(X), n_classes), dtype=float)
    seen_classes = clf.named_steps["m"].classes_
    for i, cls in enumerate(seen_classes):
        out[:, int(cls)] = raw[:, i]
    return out

def empirical_prior_from_past(y_past, n_classes):
    if len(y_past) == 0:
        return np.ones(n_classes, dtype=float) / n_classes

    counts = np.bincount(np.asarray(y_past, dtype=int), minlength=n_classes).astype(float)
    if counts.sum() == 0:
        return np.ones(n_classes, dtype=float) / n_classes
    return counts / counts.sum()

def make_time_oof_logreg_features(X_train, y_train_cls, alpha, n_splits=LOGREG_OOF_SPLITS):
    X_train = X_train.reset_index(drop=True)
    y_train_cls = np.asarray(y_train_cls, dtype=int)

    n = len(X_train)
    n_classes = int(np.max(y_train_cls)) + 1

    if n < 4 or n_classes < 2:
        prior = empirical_prior_from_past([], n_classes)
        proba = np.tile(prior, (n, 1))
        pred = np.argmax(proba, axis=1)
        return pred, proba, "past_only_empirical_priors"

    n_splits = max(2, min(n_splits, n - 1))
    tscv = TimeSeriesSplit(n_splits=n_splits)

    oof_proba = np.zeros((n, n_classes), dtype=float)
    covered = np.zeros(n, dtype=bool)

    for tr_idx, va_idx in tscv.split(X_train):
        ytr = y_train_cls[tr_idx]
        prior = empirical_prior_from_past(ytr, n_classes)

        if len(np.unique(ytr)) < 2:
            oof_proba[va_idx] = np.tile(prior, (len(va_idx), 1))
            covered[va_idx] = True
            continue

        clf = build_logreg_classifier(alpha, ytr)
        clf.fit(X_train.iloc[tr_idx], ytr)
        oof_proba[va_idx] = aligned_proba(clf, X_train.iloc[va_idx], n_classes)
        covered[va_idx] = True

    missing_idx = np.where(~covered)[0]
    for idx in missing_idx:
        prior = empirical_prior_from_past(y_train_cls[:idx], n_classes)
        oof_proba[idx] = prior

    oof_pred = np.argmax(oof_proba, axis=1)
    return oof_pred, oof_proba, "past_only_empirical_priors"

def make_logreg_meta_frame(pred_cls, pred_proba, prefix="logreg"):
    return pd.DataFrame({
        f"{prefix}_pred_class": np.asarray(pred_cls, dtype=int),
        f"{prefix}_max_proba": np.max(pred_proba, axis=1),
        f"{prefix}_entropy": normalized_entropy(pred_proba),
        f"{prefix}_margin_top1_top2": top2_margin(pred_proba),
    })

def build_logreg_augmented(train_df, infer_df, scheme_name, alpha):
    train_df = train_df.reset_index(drop=True)
    infer_df = infer_df.reset_index(drop=True)

    X_train = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    y_train = train_df[TARGET_COL].reset_index(drop=True)
    X_infer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)

    fitted_binning = fit_binning_on_train(y_train, BINNING_SCHEMES[scheme_name])
    y_cls = apply_binning(y_train, fitted_binning)

    n_classes = int(np.max(y_cls)) + 1
    if n_classes < 2:
        return None

    train_pred_cls, train_proba, fallback = make_time_oof_logreg_features(X_train, y_cls, alpha)

    if len(np.unique(y_cls)) < 2:
        infer_proba = np.tile(empirical_prior_from_past(y_cls, n_classes), (len(X_infer), 1))
    else:
        clf_full = build_logreg_classifier(alpha, y_cls)
        clf_full.fit(X_train, y_cls)
        infer_proba = aligned_proba(clf_full, X_infer, n_classes)

    infer_pred_cls = np.argmax(infer_proba, axis=1)

    train_meta = make_logreg_meta_frame(train_pred_cls, train_proba, prefix="logreg")
    infer_meta = make_logreg_meta_frame(infer_pred_cls, infer_proba, prefix="logreg")

    X_train_aug = pd.concat([X_train, train_meta], axis=1)
    X_infer_aug = pd.concat([X_infer, infer_meta], axis=1)

    meta_info = {
        "scheme": scheme_name,
        "alpha": float(alpha),
        "n_classes": int(n_classes),
        "oof_fallback": fallback,
    }
    return X_train_aug, X_infer_aug, meta_info


print("=" * 100)
print("03 LOGREG META SCREENING :: all baseline models")
print("=" * 100)
print("models =", len(MODEL_ORDER))
print("schemes =", len(BINNING_SCHEMES))
print("alphas =", len(WEIGHT_ALPHAS))
print("expected runs =", len(MODEL_ORDER) * len(BINNING_SCHEMES) * len(WEIGHT_ALPHAS))

tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
screen_rows = []
total_runs = len(MODEL_ORDER) * len(BINNING_SCHEMES) * len(WEIGHT_ALPHAS)
run_id = 0
best_so_far_by_model = {}

for model_name in MODEL_ORDER:
    base_row = baseline_summary[baseline_summary["model"] == model_name].iloc[0]

    for scheme_name in BINNING_SCHEMES:
        for alpha in WEIGHT_ALPHAS:
            fold_maes = []
            valid_run = True
            last_meta_info = None

            for tr_idx, va_idx in tscv.split(train_filt):
                fold_train = train_filt.iloc[tr_idx].copy().reset_index(drop=True)
                fold_valid = train_filt.iloc[va_idx].copy().reset_index(drop=True)

                built = build_logreg_augmented(fold_train, fold_valid, scheme_name, alpha)
                if built is None:
                    valid_run = False
                    if VERBOSE_EACH_RUN:
                        print(f"[skip] model={model_name} | scheme={scheme_name} | alpha={alpha:.1f} | reason=build_logreg_augmented returned None")
                    break

                Xtr_aug, Xva_aug, meta_info = built
                ytr = fold_train[TARGET_COL].reset_index(drop=True)
                yva = fold_valid[TARGET_COL].reset_index(drop=True)

                try:
                    est = build_regression_model(model_name)
                    est.fit(Xtr_aug, ytr)
                    pred = est.predict(Xva_aug)
                    fold_maes.append(float(mean_absolute_error(yva, pred)))
                    last_meta_info = meta_info
                except Exception as e:
                    valid_run = False
                    if VERBOSE_EACH_RUN:
                        print(f"[fail] model={model_name} | scheme={scheme_name} | alpha={alpha:.1f} | reason={type(e).__name__}: {e}")
                    break

            if not valid_run or len(fold_maes) != CV_SPLITS:
                continue

            cv_mae = float(np.mean(fold_maes))
            cv_std = float(np.std(fold_maes))

            run_id += 1
            baseline_cv = float(base_row["baseline_cv_MAE"])
            delta_cv = baseline_cv - cv_mae

            row = {
                "model": model_name,
                "scheme": scheme_name,
                "alpha": float(alpha),
                "n_classes": int(last_meta_info["n_classes"]),
                "cv_typical_MAE": round(cv_mae, 4),
                "cv_typical_std": round(cv_std, 4),
                "baseline_cv_MAE": baseline_cv,
                "delta_cv_typical": round(delta_cv, 4),
                "oof_fallback": last_meta_info["oof_fallback"],
            }
            screen_rows.append(row)

            prev_best = best_so_far_by_model.get(model_name)
            is_new_best = (
                prev_best is None
                or cv_mae < prev_best["cv_typical_MAE"]
                or (abs(cv_mae - prev_best["cv_typical_MAE"]) < 1e-12 and cv_std < prev_best["cv_typical_std"])
            )
            if is_new_best:
                best_so_far_by_model[model_name] = row

            if VERBOSE_EACH_RUN:
                fold_part = f" | folds={np.round(fold_maes, 4).tolist()}" if PRINT_FOLD_VALUES else ""
                best_tag = "  <-- NEW BEST" if is_new_best else ""
                print(
                    f"[{run_id}/{total_runs}] model={model_name} | scheme={scheme_name} | alpha={alpha:.1f} | "
                    f"cv_avg={cv_mae:.4f} | cv_std={cv_std:.4f} | baseline_cv={baseline_cv:.4f} | delta_cv={delta_cv:+.4f}"
                    f"{fold_part}{best_tag}"
                )

            if (run_id % SAVE_PROGRESS_EVERY == 0) and len(screen_rows) > 0:
                pd.DataFrame(screen_rows).sort_values(
                    ["cv_typical_MAE", "cv_typical_std", "delta_cv_typical"],
                    ascending=[True, True, False]
                ).to_csv(
                    ARTIFACT_DIR / "03_logreg_screening_all_baseline_bigsearch_optuna_progress.csv",
                    index=False
                )

    if model_name in best_so_far_by_model:
        b = best_so_far_by_model[model_name]
        print(f"\nBEST SO FAR for {model_name}: scheme={b['scheme']} | alpha={b['alpha']:.1f} | cv_avg={b['cv_typical_MAE']:.4f} | delta_cv={b['delta_cv_typical']:+.4f}\n")

logreg_screen_df = pd.DataFrame(screen_rows).sort_values(
    ["cv_typical_MAE", "cv_typical_std", "delta_cv_typical"],
    ascending=[True, True, False],
).reset_index(drop=True)

display(logreg_screen_df.head(30))
logreg_screen_df.to_csv(ARTIFACT_DIR / "03_logreg_screening_all_baseline_bigsearch_optuna_last_snapshot.csv", index=False)


In [ ]:

best_rows = []

Xtrain = train_filt.drop(columns=[TARGET_COL]).reset_index(drop=True)
ytrain = train_filt[TARGET_COL].reset_index(drop=True)

for model_name in MODEL_ORDER:
    sub = logreg_screen_df[logreg_screen_df["model"] == model_name].copy()
    if sub.empty:
        continue

    best = sub.iloc[0]
    base_row = baseline_summary[baseline_summary["model"] == model_name].iloc[0]

    built_typ = build_logreg_augmented(train_filt, holdout_typical, best["scheme"], float(best["alpha"]))
    built_full = build_logreg_augmented(train_filt, holdout_full, best["scheme"], float(best["alpha"]))

    if built_typ is None or built_full is None:
        continue

    Xtr_aug, Xhold_typ_aug, meta_info = built_typ
    _, Xhold_full_aug, _ = built_full

    est = build_regression_model(model_name)
    est.fit(Xtr_aug, ytrain)

    pred_typ = est.predict(Xhold_typ_aug)
    pred_full = est.predict(Xhold_full_aug)

    m_typ = reg_metrics(holdout_typical[TARGET_COL], pred_typ)
    m_full = reg_metrics(holdout_full[TARGET_COL], pred_full)

    best_rows.append({
        "model": model_name,
        "scheme": best["scheme"],
        "alpha": float(best["alpha"]),
        "n_classes": int(best["n_classes"]),
        "cv_typical_MAE": float(best["cv_typical_MAE"]),
        "cv_typical_std": float(best["cv_typical_std"]),
        "delta_cv_typical": float(best["delta_cv_typical"]),
        "holdout_typical_MAE": round(m_typ["MAE"], 4),
        "holdout_typical_RMSE": round(m_typ["RMSE"], 4),
        "holdout_typical_MdAE": round(m_typ["MdAE"], 4),
        "holdout_full_MAE": round(m_full["MAE"], 4),
        "holdout_full_RMSE": round(m_full["RMSE"], 4),
        "holdout_full_MdAE": round(m_full["MdAE"], 4),
        "baseline_holdout_typical_MAE": float(base_row["baseline_holdout_typical_MAE"]),
        "baseline_holdout_full_MAE": float(base_row["baseline_holdout_full_MAE"]),
        "delta_holdout_typical": round(float(base_row["baseline_holdout_typical_MAE"]) - m_typ["MAE"], 4),
        "delta_holdout_full": round(float(base_row["baseline_holdout_full_MAE"]) - m_full["MAE"], 4),
        "oof_fallback": best["oof_fallback"],
    })

logreg_best_by_model = pd.DataFrame(best_rows).sort_values(
    ["cv_typical_MAE", "holdout_typical_MAE"]
).reset_index(drop=True)

display(logreg_best_by_model)

logreg_screen_df.to_csv(ARTIFACT_DIR / "03_logreg_screening_all_baseline_bigsearch_optuna.csv", index=False)
logreg_best_by_model.to_csv(ARTIFACT_DIR / "03_logreg_best_by_model_all_baseline_bigsearch_optuna.csv", index=False)
baseline_summary.to_csv(ARTIFACT_DIR / "03_logreg_baseline_summary_all_baseline_bigsearch_optuna.csv", index=False)

protocol = {
    "notebook": "03_meta_logreg_all_baseline_bigsearch_optuna_final",
    "selection_rule": "best CV typical MAE on train_filt using TimeSeriesSplit; no holdout usage for selection",
    "external_holdout": "same 80/20 chronological holdout as baseline notebook",
    "train_definition": "train_full filtered to duration_hours < 960",
    "holdout_typical_definition": "holdout_full filtered to duration_hours < 960",
    "holdout_full_definition": "full external holdout",
    "model_source": "exact baseline pipelines from ML_razdel_pointfix_final.ipynb",
    "models": MODEL_ORDER,
    "num_schemes": len(BINNING_SCHEMES),
    "num_alphas": len(WEIGHT_ALPHAS),
    "expected_runs": len(MODEL_ORDER) * len(BINNING_SCHEMES) * len(WEIGHT_ALPHAS),
}
with open(ARTIFACT_DIR / "03_logreg_protocol_all_baseline_bigsearch_optuna.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, ensure_ascii=False, indent=2)

print("Saved:")
for name in [
    "03_logreg_screening_all_baseline_bigsearch_optuna.csv",
    "03_logreg_best_by_model_all_baseline_bigsearch_optuna.csv",
    "03_logreg_baseline_summary_all_baseline_bigsearch_optuna.csv",
    "03_logreg_protocol_all_baseline_bigsearch_optuna.json",
]:
    print("-", ARTIFACT_DIR / name)
